# Multi-modal Fine-tuning - Audio + Text

Built by [Trelis.com](https://Trelis.com). Find Trelis on [HuggingFace](https://huggingface.co/Trelis) and [GitHub](https://github.com/TrelisResearch).

This is a private notebook - available for purchase from Trelis.com .

---

## Getting Set Up
### Setup on an Ampere GPU (A40, A6000, A100, H100) with Cuda 12.1 and Pytorch 2.2.1 - RECOMMENDED.
Ampere architecture GPUs allow for the use of Flash Attention, which provides a speed up. Otherwise, you need to train with fp16 instead of bf16.

For the best reproducibility, run this script on an A40:
- Runpod one-click template [here](https://runpod.io/gsc?template=ifyqsvjlzj&ref=jmfkcdio) - easier setup.
- Vast.ai one-click template [here](https://cloud.vast.ai/?ref_id=98762&creator_id=98762&name=Fine-tuning%20Notebook%20by%20Trelis%20-%20Cuda%2012.1) - offers smaller GPUs (which are cheaper to run).

### GPU requirements for training

- Probably 20+ GB of VRAM for fine-tuning in 16 bits. You could alternatively add in bitsandbytes and do QLoRA fine-tuning.

## Set up Trelis Assistant for Jupyter Lab (optional)

In [ ]:
import os
import subprocess

def check_node_yarn():
    # Check if Node.js is installed
    try:
        node_version = subprocess.check_output(["node", "--version"]).decode().strip()
        print(f"Node.js is installed: {node_version}")
    except FileNotFoundError:
        print("Node.js is not installed.")
        install_node()

    # Check if Yarn is installed
    try:
        yarn_version = subprocess.check_output(["yarn", "--version"]).decode().strip()
        print(f"Yarn is installed: {yarn_version}")
    except FileNotFoundError:
        print("Yarn is not installed.")
        install_yarn()

def install_node():
    print("Installing Node.js...")
    # Download Node.js binary (you can change the version if needed)
    os.system("curl -O https://nodejs.org/dist/v22.6.0/node-v22.6.0-linux-x64.tar.xz --silent")
    
    # Extract Node.js
    os.system("tar -xf node-v22.6.0-linux-x64.tar.xz --no-same-owner")
    
    # Add Node.js to the PATH for the current notebook session
    node_bin_path = os.path.abspath("node-v22.6.0-linux-x64/bin")
    os.environ['PATH'] = node_bin_path + ":" + os.environ['PATH']
    print(f"Node.js installed to {node_bin_path}")

    # Verify installation
    node_version = subprocess.check_output(["node", "--version"]).decode().strip()
    print(f"Node.js version: {node_version}")

def install_yarn():
    print("Installing Yarn...")
    # Install Yarn via script
    os.system("curl -o- -L https://yarnpkg.com/install.sh | bash")
    
    # Add Yarn to the PATH for the current notebook session
    yarn_bin_path = os.path.expanduser("~/.yarn/bin")
    os.environ['PATH'] = yarn_bin_path + ":" + os.environ['PATH']
    print(f"Yarn installed to {yarn_bin_path}")

    # Verify installation
    yarn_version = subprocess.check_output(["yarn", "--version"]).decode().strip()
    print(f"Yarn version: {yarn_version}")

# Run the check and installation process
check_node_yarn()

In [ ]:
!pip install trelis

# Then REFRESH the page and click on Trelis in the right-hand bar

# Setup

We first setup the environment with the primary necessary libraries and login into Hugging Face.

In [ ]:
!python -m pip install --upgrade pip -q
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -qU accelerate datasets peft bitsandbytes hf_transfer tensorboard requests matplotlib librosa

# Can be a good idea to re-start the kernel after this

In [ ]:
# !pip freeze > requirements.txt

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

# Loading the model and the dataset

We load the model from the Hugging Face hub. `idefics2-8b` has gone through instruction fine-tuning on a large mixture of multimodal datasets and as such is a strong starting-point to fine-tune on your own use-case. We will start from this checkpoint.

Fine-tuning will be done on a small chess dataset.

In [ ]:
# Enable fast weights download and upload
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
# os.environ["CUDA_VISIBLE_DEVICES"] = "0" # if you're connected to multiple gpus but only want to use one.

In [ ]:
# Qwen can have issues with flash attention so we won't use it.
# !pip uninstall flash-attn -y

In [ ]:
import torch, gc
from PIL import Image
from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

debug=False

device = 'cuda'

model_id = "Qwen/Qwen2-Audio-7B-Instruct"

torch_dtype = torch.bfloat16 # or float16 for T4/colab

# # Define the quantization configuration with NF4 and double quantization
# quant_config = BitsAndBytesConfig(
#     load_in_4bit=True,             # Use 4-bit quantization (NF4)
#     bnb_4bit_quant_type="nf4",     # Set quantization type to NF4
#     bnb_4bit_use_double_quant=True # Enable double quantization
# )

# del model
# gc.collect()

model = Qwen2AudioForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch_dtype, #float16 for colab
    # attn_implementation="flash_attention_2", # do not use for colab and T4. Do not use for quantization either. Not using for Qwen.
    trust_remote_code=True,
    # quantization_config=quant_config, # for colab 
).to(device)
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

In [ ]:
# # Function to print dtypes of model modules
# def print_module_dtypes(model):
#     print("Module dtypes:")
#     for name, module in model.named_modules():
#         try:
#             dtype = next(module.parameters()).dtype
#         except StopIteration:  # No parameters in the module
#             dtype = "No parameters"
#         print(f"{name}: {dtype}")

# # Function to print dtypes of model parameters
# def print_parameter_dtypes(model):
#     print("\nParameter dtypes:")
#     for name, param in model.named_parameters():
#         print(f"{name}: {param.dtype}")

# # Print the dtypes
# print_module_dtypes(model)
# print_parameter_dtypes(model)

In [ ]:
# print(model.config)

In [ ]:
# print(model)

In [ ]:
from IPython.display import Audio, display
from urllib.request import urlopen
from io import BytesIO
import librosa

TASK_PROMPT = "What type of bird do you hear?"

def run_example(audio_url):
    """
    Runs inference on an audio file and prints the result.
    """
    try:
        # Load audio
        audio_data, sampling_rate = librosa.load(BytesIO(urlopen(audio_url).read()), sr=None)

        # Play audio
        display(Audio(audio_data, rate=sampling_rate))

        # Prepare conversation
        conversation = [
            {"role": "user", "content": [
                {"type": "audio", "audio_url": audio_url},
                {"type": "text", "text": TASK_PROMPT},
            ]}
        ]
        
        # Process conversation
        text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)

        if debug:
            print(f"Templated text:\n{text}")
        
        audios = []
        
        for message in conversation:
            if isinstance(message["content"], list):
                for ele in message["content"]:
                    if ele["type"] == "audio" and "audio_url" in ele:
                        try:
                            
                            # Confirm the sampling rate
                            sr = int(processor.feature_extractor.sampling_rate)
                            
                            # Load and append audio
                            audio_data, sr_loaded = librosa.load(
                                BytesIO(urlopen(ele["audio_url"]).read())
                            )
                            from librosa import resample

                            # Resample audio data to match the processor's sampling rate
                            audio_data = resample(audio_data, orig_sr=sampling_rate, target_sr=sr)
                            if debug:
                                print(f"Loaded with sample rate {sr_loaded}")
                            
                            audios.append(audio_data)
                        except Exception as e:
                            print(f"Failed to process audio from {ele['audio_url']}: {e}")


        inputs = processor(
            text=text,
            audios=audios,
            return_tensors="pt",
            padding=True,
            truncation=True,
            # max_length=512
        ).to("cuda")

        generate_ids = model.generate(**inputs, max_new_tokens=256)

        print(f"Input shape: {inputs.input_ids.size(1)}, Generated shape: {generate_ids.shape}")
        
        generate_ids = generate_ids[:, inputs.input_ids.size(1):]
        response = processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

        print(response)

    except Exception as e:
        print(f"An error occurred: {e}")

# Example usage:  Replace with your actual audio URL
audio_url = "https://xeno-canto.org/841748/download" #Example: "https://www.example.com/birdsong.wav"
run_example(audio_url)

In [ ]:
print(f"Sampling rate type: {type(processor.feature_extractor.sampling_rate)}")
print(f"Sampling rate value: {processor.feature_extractor.sampling_rate}")

In [ ]:
print(processor.feature_extractor.sampling_rate)

## Load Dataset

In [ ]:
from datasets import load_dataset

# load and prepare dataset
ds = load_dataset("Trelis/bird-songs") # to train on bounding boxes

train_dataset = ds["train"]
eval_dataset = ds["validation"]

In [ ]:
train_dataset[0]

# Evaluation before Training

In [ ]:
import torch
from PIL import Image
from torchvision.transforms.functional import to_pil_image, resize

def run_model_evaluation(dataset):
    for example in dataset:
        audio = example["url"]

        results = run_example(audio)
        print(f"Ground truth answer: {example['name']}")
        print("\n")

    return

In [ ]:
# Usage
run_model_evaluation(eval_dataset)
# print(eval_results)

# Training loop

We first define the data collator which takes list of samples and return input tensors fed to the model. There are 4 tensors types we are interested:
- `input_ids`: these are the input indices fed to the language model
- `attention_mask`: the attention mask for the `input_ids` in the language model


In [ ]:
print(processor.tokenizer.bos_token)
print(processor.tokenizer.bos_token_id)
print("---")

print(processor.tokenizer.eos_token)
print(processor.tokenizer.eos_token_id)
print("---")
print(processor.tokenizer.pad_token_id)

In [ ]:
# print(processor.tokenizer)

In [ ]:
# Test processing

audios = [librosa.load(
    BytesIO(urlopen(audio_url).read()), # defined above
    sr=processor.feature_extractor.sampling_rate)[0]]

batch = processor(text="Test", audios=audios, return_tensors="pt", padding=True)
print(batch)

## Data Collator

In [ ]:
print(processor.feature_extractor.n_samples)

In [ ]:
import torch
from torch.utils.data import DataLoader
import librosa
import numpy as np
from io import BytesIO
from urllib.request import urlopen
from typing import List, Dict, Any

debug = False

class AudioDataCollator:
    def __init__(self, processor, task_prompt: str, max_length: int = None):
        self.processor = processor
        self.task_prompt = task_prompt
        self.max_length = max_length or processor.feature_extractor.n_samples
        self.sampling_rate = processor.feature_extractor.sampling_rate

    def process_audio(self, audio_url: str) -> np.ndarray:
        """Process a single audio file."""
        try:
            # Load audio from URL
            audio, sr = librosa.load(
                BytesIO(urlopen(audio_url).read()),
                sr=self.sampling_rate
            )
            
            # Truncate or pad audio
            if len(audio) > self.max_length:
                if debug:
                    print(f"Audio length ({len(audio)}) exceeds max_length ({self.max_length}). Truncating...")
                audio = audio[:self.max_length]
            # else: # better to allow for padding in the processor, perhaps
            #     audio = np.pad(audio, (0, self.max_length - len(audio)), mode='constant')
            
            return audio.astype(np.float32)  # Ensure float32 for compatibility
        except Exception as e:
            raise RuntimeError(f"Error processing audio from {audio_url}: {str(e)}")

    def __call__(self, examples: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        valid_examples = []
        audios = []
        combined_texts = []  # Combined prompt and caption for decoder-only model

        # Process each example
        for example in examples:
            try:
                # Process audio
                audio = self.process_audio(example["url"])
                audios.append(audio)

                # Combine prompt and caption into a single input sequence
                conversation = [
                    {"role": "user", "content": [
                        {"type": "audio", "audio_url": example["url"]},
                        {"type": "text", "text": self.task_prompt}
                    ]},
                    {"role": "assistant", "content": example["name"]}  # Caption
                ]
                combined_text = self.processor.apply_chat_template(
                    conversation, add_generation_prompt=False, tokenize=False
                )
                combined_texts.append(combined_text)

                valid_examples.append(example)
            except Exception as e:
                print(f"Failed to process {example['id']}: {str(e)}")
                continue

        if not valid_examples:
            raise ValueError("No valid examples in batch")

        if debug:    
            # Debugging: Validate inputs
            print(f"\n=== Debugging Inputs ===")
            print(f"Number of combined texts: {len(combined_texts)}")
            print(f"Number of audios: {len(audios)}")

        # Process combined inputs with the processor
        try:
            inputs = self.processor(
                text=list(combined_texts),
                audios=list(audios),
                return_tensors="pt",
                padding=True
            )
        except Exception as e:
            print(f"Processor error: {str(e)}")
            raise

        # Prepare labels for decoder-only setup
        labels = inputs["input_ids"].clone()
        
        # Debugging: Print the input IDs and tokenized data
        if debug:
            print("\n=== Debugging Inputs ===")
            print(f"Input IDs:\n{inputs['input_ids']}")
            print(f"Input IDs Shape: {inputs['input_ids'].shape}")
            print(f"Input IDs Type: {type(inputs['input_ids'])}")
        
        # Mask the prompt portion dynamically for each example
        for i in range(len(combined_texts)):
            try:
                # Ensure we handle indexing correctly
                input_ids_row = inputs["input_ids"][i]
                if debug:
                    print(f"\nProcessing example {i}")
                    print(f"Input IDs for this example: {input_ids_row}")
        
                # Find the tokenized sequence for "<|im_start|>assistant"
                assistant_start_tokens = self.processor.tokenizer.encode("<|im_start|>assistant", add_special_tokens=False)
                if debug:
                    print(f"Assistant Start Token IDs: {assistant_start_tokens}")
        
                # Search for the sequence of tokens in the input
                assistant_start_idx = -1
                for j in range(len(input_ids_row) - len(assistant_start_tokens) + 1):
                    if (input_ids_row[j : j + len(assistant_start_tokens)] == torch.tensor(assistant_start_tokens, device=input_ids_row.device)).all():
                        assistant_start_idx = j
                        break
        
                if debug:
                    print(f"Assistant Start Index: {assistant_start_idx}")
        
                if assistant_start_idx != -1:  # Check if the sequence exists in the input
                    # Mask everything before the start of the assistant's response
                    labels[i, :assistant_start_idx + len(assistant_start_tokens)] = -100
                else:
                    # Fallback if the sequence is not found
                    print(f"Warning: '<|im_start|>assistant' not found in input for example {i}.")
                    labels[i, :] = -100  # Mask entire sequence if the sequence is missing
            except Exception as e:
                print(f"Error processing example {i}: {str(e)}")
        
        if debug:
            # Debugging: Check shapes and labels
            print("\n=== Debugging Processor Outputs ===")
            print(f"Input IDs Shape: {inputs['input_ids'].shape}")
            print(f"Attention Mask Shape: {inputs['attention_mask'].shape}")
            print(f"Labels Shape: {labels.shape}")
            print(f"First label row (masked): {labels[0]}")


        return {
            "input_ids": inputs.input_ids,
            "attention_mask": inputs.attention_mask,
            "input_features": inputs.input_features,
            "feature_attention_mask": inputs.feature_attention_mask,
            "labels": labels
        }

def inspect_batch(batch: Dict[str, torch.Tensor], processor) -> None:
    """Detailed inspection of a batch, including label decoding."""
    print("\n=== Detailed Batch Inspection ===")
    
    for key, tensor in batch.items():
        print(f"\n{key}:")
        print(f"  Shape: {tensor.shape}")
        print(f"  Type: {tensor.dtype}")
        print(f"  Device: {tensor.device}")
        if key not in ['input_features', 'feature_attention_mask']:
            print(f"  First row: {tensor[0].tolist()}")

    # Decode labels and print the first one
    if "input_ids" in batch:
        print("\n=== Decoded input_ids ===")
        first_input = batch["input_ids"][0].tolist()
        decoded_input = processor.tokenizer.decode(first_input, skip_special_tokens=False)
        print(f"Decoded Input (First Row):\n{decoded_input}")

    # Decode labels and print the first one
    if "labels" in batch:
        print("\n=== Decoded Labels ===")
        first_label = batch["labels"][0].tolist()
        # Filter out the masked token (-100) values
        valid_tokens = [token for token in first_label if token != -100]
        decoded_label = processor.tokenizer.decode(valid_tokens, skip_special_tokens=False)
        print(f"Decoded Label (First Row):\n{decoded_label}")

In [ ]:
print(processor.tokenizer.encode("<|im_start|>assistant"))

In [ ]:
# Initialize with debugging
data_collator = AudioDataCollator(
    processor=processor,
    task_prompt="Identify the bird species in this audio clip:"
)

# Create dataloader
dataloader = DataLoader(
    train_dataset,
    batch_size=1,
    collate_fn=data_collator,
    shuffle=True
)

# Test first batch with detailed inspection
for batch in dataloader:
    inspect_batch(batch, processor)
    break

## Prepare LoRA and Trainer

In [ ]:
print(model)

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=32,                 # Rank (usually 8, 16, or 32 depending on model size and needs)
    lora_alpha=32,         # Scaling factor for the low-rank updates
    use_rslora=True,
    # target_modules=[
    #     "k_proj","q_proj","v_proj","out_proj",
    #     "up_proj","down_proj","gate_proj",
    #     "fc1","fc2",
    #     "linear"
    # ],
    target_modules="all-linear",
    # modules_to_save=["lm_head","embed_tokens"], # optionally train the embeddings
    lora_dropout=0.1,      # Dropout for low-rank adapter layers
    bias="none",           # Bias in adapter layers: "none", "all", or "lora_only"
    task_type="CAUSAL_LM"  # Task type: "CAUSAL_LM", "SEQ_2_SEQ_LM", or "TOKEN_CLS"
)

In [ ]:
from peft import get_peft_model

model=get_peft_model(model,lora_config)

In [ ]:
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments, Trainer

lr=2e-5

# for main fine-tuning
epochs=1
schedule="constant"

# # Optional, run this again on the fine-tuned model to do an annealing phase
# epochs=1
# schedule="cosine"

batch_size = 4

run_name=f"trelis-song-birds-{lr}_lr-{epochs}_epochs-{schedule}_schedule"

training_args = TrainingArguments(
    # max_steps=1,
    num_train_epochs=epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=1,
    # warmup_steps=50, #comment in only if you have a lot more than 50 samples.
    learning_rate=lr,
    weight_decay=0.01,
    logging_steps=0.1,
    output_dir="fine-tuned-model",
    eval_strategy="steps",
    eval_steps=0.2,
    lr_scheduler_type=schedule,
    # save_strategy="steps",
    # save_steps=250,
    # save_total_limit=1,
    # fp16=True, #if using Colab, but then you need to use bitsandbytes quantization too.
    bf16=True,
    hub_model_id="Trelis/Qwen-song-birds",
    remove_unused_columns=False,
    report_to="tensorboard",
    run_name=run_name,
    logging_dir=f"./logs/{run_name}",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant':False} # causes issues to set True for Qwen, although this will cause slow-down
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset, # You can also evaluate (loss) on the eval set, note that it will incur some additional GPU memory
)

# TIP
# It can be nice to train with a constant learning rate, keep repeating that til the eval loss is flattening, then train for some steps with linear decay/annealing.

# Training and pushing to the hub

The training can take a few minutes depending on the hardware you use.

In [ ]:
trainer.train()

# Evaluation

Let's evaluate the model. First, we can have a look at a qualitative generation from the model.

In [ ]:
model.eval()
run_model_evaluation(eval_dataset)

In [ ]:
stop here before pushing to hub.

# Push to Hub

In [ ]:
from huggingface_hub import login
login()

In [ ]:
trainer.push_to_hub()

In [ ]:
# Merge the model
model = model.merge_and_unload()

In [ ]:
model.push_to_hub("Trelis/song-birds")
processor.push_to_hub("Trelis/song-birds")